# 🧠 MemMachine × llama.cpp デモ（Docker不使用）

```
ユーザー（このノートブック）
       ↓  memmachine-client SDK
MemMachine サーバー  （ポート 8765）
  ├─ PostgreSQL  → プロフィール / セマンティックメモリ
  └─ Neo4j       → エピソードメモリ（グラフDB）
       ↓
llama-cpp-python サーバー × 2
  ├─ LLM サーバー       （ポート 9081）
  └─ Embedding サーバー （ポート 9082）
```

> ⚡ GPU ランタイム推奨だが、CPU でも動作します  
> ⚠️ ポート **8080** は実行環境によってシステムが使用する場合があるため使用しません

---

## 📦 Step 1: 環境確認（GPU / CPU 自動判定）

In [ ]:
import subprocess, sys, os, time, json, textwrap, re, requests

def sh(cmd, check=True, quiet=False):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if not quiet:
        if r.stdout.strip(): print(r.stdout.strip())
        if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}\n{r.stderr}")
    return r.stdout.strip()

try:
    gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", quiet=True)
    print(f"🎮 GPU: {gpu}")
    HAS_GPU = True
except:
    print("⚠️  GPU 未検出 — CPU モードで動作します（推論が遅くなります）")
    HAS_GPU = False

CUDA_VER = "cpu"
if HAS_GPU:
    try:
        m = re.search(r"release (\d+)\.(\d+)", sh("nvcc --version", quiet=True))
        if m: CUDA_VER = f"cu{m.group(1)}{m.group(2)}"
    except:
        try:
            m = re.search(r"CUDA Version: (\d+)\.(\d+)", sh("nvidia-smi", quiet=True))
            if m: CUDA_VER = f"cu{m.group(1)}{m.group(2)}"
        except: pass

N_GPU = -1 if HAS_GPU else 0

# ポート定義
# 8080 は実行環境によってシステムが予約している場合があるため使用しない
MM_PORT  = 8765  # MemMachine
LLM_PORT = 9081  # llama.cpp LLM
EMB_PORT = 9082  # llama.cpp Embedding

print(f"🔧 CUDA: {CUDA_VER}  N_GPU_LAYERS: {N_GPU}")
print(f"🐍 Python: {sys.version.split()[0]}")
print(f"🔌 ポート: MemMachine={MM_PORT}, LLM={LLM_PORT}, Embed={EMB_PORT}")
if not HAS_GPU:
    print("""
⚠️  CPU モード注意事項:
  - モデル読み込みに 1〜3 分かかります
  - 1回の推論に 数分〜10分以上 かかる場合があります
  - タイムアウトを長めに設定します
""")

## 📦 Step 2: パッケージのインストール

In [ ]:
# PORT/HOST 環境変数をクリア（llama.cpp との干渉を防ぐ）
for k in ["PORT", "HOST"]: os.environ.pop(k, None)

INDEX_URL = f"https://abetlen.github.io/llama-cpp-python/whl/{CUDA_VER}"
print(f"📦 llama-cpp-python: {INDEX_URL}")
print(f"   モード: {'GPU (CUDA)' if HAS_GPU else 'CPU'}")

!pip install -q "llama-cpp-python[server]" --extra-index-url {INDEX_URL}
!pip install -q --upgrade "llama-cpp-python[server]" --extra-index-url {INDEX_URL}
!pip install -q memmachine-server memmachine-client huggingface_hub requests

v = subprocess.run("pip show llama-cpp-python | grep Version",
                   shell=True, capture_output=True, text=True).stdout.strip()
print(f"✅ {v}")

## 🗄️ Step 3: Neo4j のインストール & 起動

In [ ]:
%%bash
set -e
# Neo4j 公式リポジトリを追加してインストール
wget -q -O - https://debian.neo4j.com/neotechnology.gpg.key | apt-key add - 2>/dev/null
echo "deb https://debian.neo4j.com stable 5" > /etc/apt/sources.list.d/neo4j.list
apt-get update -qq
apt-get install -y -qq neo4j 2>&1 | tail -5
which neo4j && neo4j --version
neo4j-admin dbms set-initial-password neo4j_pass 2>/dev/null || true
neo4j start
echo "✅ 完了"


In [ ]:
print("⏳ Neo4j 起動待機...", end="")
for _ in range(30):
    try:
        r = requests.get("http://localhost:7474", timeout=3)
        if r.status_code in (200, 401): print(" ✅ 起動完了"); break
    except: pass
    print(".", end="", flush=True); time.sleep(3)
else:
    print(" ❌ タイムアウト")

## 🗄️ Step 4: PostgreSQL + pgvector のインストール

In [ ]:
%%bash
set -e
apt-get install -y -qq postgresql postgresql-contrib
apt-get install -y -qq postgresql-server-dev-all build-essential git

# pgvector をソースからビルド
cd /tmp && rm -rf pgvector
git clone --depth 1 https://github.com/pgvector/pgvector.git
cd pgvector && make -j$(nproc) && make install

# PostgreSQL 起動
PG_VER=$(ls /usr/lib/postgresql/)
pg_ctlcluster $PG_VER main start || true
sleep 3

# ユーザー・DB・拡張の作成
su -s /bin/bash postgres -c "psql -c \"CREATE USER memmachine WITH PASSWORD 'memmachine_pass';\"" 2>/dev/null || true
su -s /bin/bash postgres -c "psql -c \"CREATE DATABASE memmachine OWNER memmachine;\"" 2>/dev/null || true
su -s /bin/bash postgres -c "psql -d memmachine -c \"CREATE EXTENSION IF NOT EXISTS vector;\"" 2>/dev/null || true

# TCP 接続を許可
CONF=$(su -s /bin/bash postgres -c "psql -t -c 'SHOW config_file;'" | tr -d ' \n')
HBA=$(su -s /bin/bash postgres -c "psql -t -c 'SHOW hba_file;'" | tr -d ' \n')
echo "listen_addresses = 'localhost'" >> $CONF
echo "host all all 127.0.0.1/32 md5" >> $HBA
pg_ctlcluster $PG_VER main reload
sleep 2

PGPASSWORD=memmachine_pass psql -h 127.0.0.1 -U memmachine -d memmachine -c "SELECT 'OK';"
echo "✅ PostgreSQL + pgvector 完了"


## 📥 Step 5: GGUF モデルのダウンロード

In [ ]:
from huggingface_hub import hf_hub_download
os.makedirs("/content/models", exist_ok=True)

print("📥 LLM (Llama-3.2-1B-Instruct Q4_K_M ≈ 0.8 GB)...")
LLM_PATH = hf_hub_download(
    repo_id="bartowski/Llama-3.2-1B-Instruct-GGUF",
    filename="Llama-3.2-1B-Instruct-Q4_K_M.gguf",
    local_dir="/content/models"
)
print(f"✅ {LLM_PATH}")

print("\n📥 Embedding (nomic-embed-text-v1.5 Q4_K_M ≈ 80 MB)...")
EMB_PATH = hf_hub_download(
    repo_id="nomic-ai/nomic-embed-text-v1.5-GGUF",
    filename="nomic-embed-text-v1.5.Q4_K_M.gguf",
    local_dir="/content/models"
)
print(f"✅ {EMB_PATH}")

for f in os.listdir("/content/models"):
    mb = os.path.getsize(f"/content/models/{f}") / 1e6
    print(f"  📄 {f}  ({mb:.0f} MB)")

## 🚀 Step 6: llama-cpp-python サーバーの起動

- ポート **9081** : LLM（テキスト生成）
- ポート **9082** : Embedding
- GPU/CPU は Step 1 の判定結果を自動使用

In [ ]:
# PORT/HOST を除いた環境変数セットを用意（llama.cpp との干渉を防ぐ）
env_llamacpp = {k: v for k, v in os.environ.items() if k not in ("PORT", "HOST")}

def tail_log(path, n=20):
    try: return "".join(open(path).readlines()[-n:])
    except: return "(ログ未生成)"

def launch_llamacpp(model_path, port, name, embedding=False):
    subprocess.run(f"fuser -k {port}/tcp", shell=True, capture_output=True)
    time.sleep(2)
    log_path = f"/content/{name}.log"
    cmd = [
        "python", "-m", "llama_cpp.server",
        "--model",        str(model_path),
        "--host",         "127.0.0.1",
        "--port",         str(port),
        "--n_gpu_layers", str(N_GPU),
        "--n_ctx",        "2048",
    ]
    if embedding: cmd += ["--embedding", "true"]
    proc = subprocess.Popen(
        cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT, env=env_llamacpp
    )
    print(f"🚀 {name} PID={proc.pid} port={port} ({'GPU' if HAS_GPU else 'CPU'})")
    wait_min = 15 if not HAS_GPU else 10
    print(f"   ⏳ 起動待機 (最大{wait_min}分)", end="", flush=True)
    for i in range(wait_min * 12):
        time.sleep(5)
        if proc.poll() is not None:
            print(f"\n   ❌ 終了 (code={proc.returncode})")
            print(tail_log(log_path)); return None
        try:
            r = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=3)
            if r.status_code == 200:
                mid = r.json()["data"][0]["id"]
                print(f" ✅ {i*5}s (model={mid})"); return proc
        except: pass
        if i % 6 == 5:
            print(f"\n   [{i*5}s] {tail_log(log_path, 1).strip()}", end="", flush=True)
        else:
            print(".", end="", flush=True)
    print(f"\n   ❌ タイムアウト"); print(tail_log(log_path)); return None

# LLM → Embedding の順で逐次起動（メモリ競合を避ける）
proc_llm = launch_llamacpp(LLM_PATH, LLM_PORT, "llamacpp_llm")
if proc_llm:
    proc_emb = launch_llamacpp(EMB_PATH, EMB_PORT, "llamacpp_embed", embedding=True)
else:
    proc_emb = None; print("⚠️ LLM 起動失敗のため Embedding をスキップ")

if proc_llm and proc_emb:
    LLM_MODEL_ID = requests.get(f"http://127.0.0.1:{LLM_PORT}/v1/models").json()["data"][0]["id"]
    EMB_MODEL_ID  = requests.get(f"http://127.0.0.1:{EMB_PORT}/v1/models").json()["data"][0]["id"]
    print(f"\nLLM モデル ID : {LLM_MODEL_ID}")
    print(f"EMB モデル ID : {EMB_MODEL_ID}")

## ⚙️ Step 7: MemMachine 設定ファイルの生成

In [ ]:
import nltk
for corpus in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(corpus, quiet=True)

os.makedirs("/content/memmachine", exist_ok=True)

cfg = f"""
logging:
  path: /content/memmachine/mem-machine.log
  level: info

episode_store:
  database: postgres_db

episodic_memory:
  long_term_memory:
    embedder: llamacpp_embedder
    reranker: bm25_reranker
    vector_graph_store: neo4j_db
  short_term_memory:
    llm_model: llamacpp_llm
    message_capacity: 200

semantic_memory:
  llm_model: llamacpp_llm
  embedding_model: llamacpp_embedder
  database: postgres_db
  config_database: postgres_db

session_manager:
  database: postgres_db

resources:
  databases:
    postgres_db:
      provider: postgres
      config:
        host: localhost
        port: 5432
        user: memmachine
        password: memmachine_pass
        db_name: memmachine

    neo4j_db:
      provider: neo4j
      config:
        uri: "bolt://localhost:7687"
        username: neo4j
        password: neo4j_pass

  embedders:
    llamacpp_embedder:
      provider: openai
      config:
        model: "{EMB_MODEL_ID}"
        api_key: "no-key"
        base_url: "http://127.0.0.1:{EMB_PORT}/v1"
        dimensions: 768

  language_models:
    llamacpp_llm:
      provider: openai-chat-completions
      config:
        model: "{LLM_MODEL_ID}"
        api_key: "no-key"
        base_url: "http://127.0.0.1:{LLM_PORT}/v1"

  rerankers:
    bm25_reranker:
      provider: bm25
""".strip()

with open("/content/memmachine/cfg.yml", "w") as f:
    f.write(cfg)
print("✅ cfg.yml 作成完了")
print(cfg)

## 🏁 Step 8: MemMachine サーバーの起動

**ポイント:** MemMachine は環境変数 `PORT` を優先して読み込む仕様のため、  
`PORT={MM_PORT}` を明示的に設定した専用の環境変数セットで起動します。

In [ ]:
# MemMachine 専用の環境変数セットを作成
# PORT を明示的に MM_PORT (8765) に設定することで、
# システムの PORT=8080 より優先させる
mm_env = {k: v for k, v in os.environ.items() if k not in ("PORT", "HOST")}
mm_env["MEMORY_CONFIG"] = "/content/memmachine/cfg.yml"
mm_env["OPENAI_API_KEY"] = "no-key"
mm_env["PORT"] = str(MM_PORT)   # ← MemMachine の server.port をここで決定

print(f"🔧 PORT={mm_env['PORT']} で起動します")

subprocess.run(f"fuser -k {MM_PORT}/tcp", shell=True, capture_output=True)
time.sleep(2)

mm_log = open("/content/memmachine/memmachine.log", "w")
proc_mm = subprocess.Popen(
    ["/usr/local/bin/memmachine-server"],
    stdout=mm_log, stderr=subprocess.STDOUT,
    cwd="/content/memmachine", env=mm_env
)
print(f"🚀 MemMachine PID={proc_mm.pid}")

print("⏳ 起動待機...", end="", flush=True)
for _ in range(60):
    time.sleep(3)
    if proc_mm.poll() is not None:
        print("\n❌ プロセス終了")
        log = open("/content/memmachine/memmachine.log").read()
        for line in log.split("\n"):
            if any(k in line for k in ["ERROR", "Error", "Exception", "Traceback", "bind"]):
                print(line)
        break
    try:
        r = requests.get(f"http://127.0.0.1:{MM_PORT}/api/v2/health", timeout=3)
        if r.status_code == 200:
            print(f" ✅ 起動完了 (port={MM_PORT})")
            print(r.json()); break
    except: pass
    print(".", end="", flush=True)
else:
    print("\n❌ タイムアウト")
    print(open("/content/memmachine/memmachine.log").read()[-2000:])

## 🔍 Step 9: llama.cpp サーバー単体テスト

In [ ]:
INFER_TIMEOUT = 300 if not HAS_GPU else 60

print("💬 LLM 推論テスト...")
resp = requests.post(f"http://127.0.0.1:{LLM_PORT}/v1/chat/completions", json={
    "model": LLM_MODEL_ID,
    "messages": [{"role": "user", "content": "日本の首都を1文で教えてください。"}],
    "max_tokens": 60
}, timeout=INFER_TIMEOUT).json()
print("  応答:", resp["choices"][0]["message"]["content"])

print("\n🔢 Embedding テスト...")
resp = requests.post(f"http://127.0.0.1:{EMB_PORT}/v1/embeddings", json={
    "model": EMB_MODEL_ID, "input": "東京は日本の首都です"
}, timeout=60).json()
vec = resp["data"][0]["embedding"]
print(f"  次元数: {len(vec)}, 先頭5要素: {[round(v, 4) for v in vec[:5]]}")

## 🧠 Step 10: MemMachine クライアント接続 & メモリ保存

In [ ]:
from memmachine_client import MemMachineClient

CLIENT_TIMEOUT = 600 if not HAS_GPU else 300
print(f"🔧 クライアント timeout: {CLIENT_TIMEOUT}s ({'CPU' if not HAS_GPU else 'GPU'} モード)")

client = MemMachineClient(base_url=f"http://127.0.0.1:{MM_PORT}", timeout=CLIENT_TIMEOUT)
project = client.get_or_create_project(org_id="demo_org", project_id="llamacpp_demo")
print("✅ プロジェクト:", project)

memory_alice = project.memory(
    group_id="default", agent_id="travel_agent",
    user_id="alice", session_id="session_001"
)

print("\n💾 メモリ追加中...")
for text, meta in [
    ("私はアリスです。東京に住んでいます。",               {"category": "profile"}),
    ("飛行機では必ず窓側の席を予約します。",               {"category": "travel_pref"}),
    ("好きな食べ物はラーメンと寿司です。",                 {"category": "food"}),
    ("2024年3月にパリを訪問し、エッフェル塔に登りました。", {"category": "past_trip"}),
    ("次の旅行先はニューヨークを検討しています。",          {"category": "future_plan"}),
]:
    result = memory_alice.add(text, metadata=meta)
    uid = result[0].uid if result else "N/A"
    print(f"  ✅ '{text[:28]}' → uid={uid}")

## 🔎 Step 11: メモリ検索テスト

In [ ]:
def get_memories(mem, query):
    """long_term + short_term 両方からメモリを取得する"""
    results = mem.search(query)
    items = []
    try:
        for ep in results.content.episodic_memory.long_term_memory.episodes:
            items.append(ep.content)
    except: pass
    try:
        for ep in results.content.episodic_memory.short_term_memory.episodes:
            if ep.content not in items: items.append(ep.content)
    except: pass
    return items

for q in ["アリスの旅行の好みは？", "アリスが過去に訪れた場所は？", "アリスの好きな食べ物は？"]:
    print(f"\n🔍 '{q}'")
    for m in get_memories(memory_alice, q)[:2]:
        print(f"  📌 {m}")

## 🤖 Step 12: メモリを活用した会話エージェント

MemMachine からメモリを取得 → llama.cpp LLM のプロンプトに注入 → 応答生成

In [ ]:
INFER_TIMEOUT = 300 if not HAS_GPU else 60

def chat_with_memory(user_question, user_id="alice"):
    mem = project.memory(
        group_id="default", agent_id="travel_agent",
        user_id=user_id, session_id="session_001"
    )
    memory_context = get_memories(mem, user_question)

    system = "あなたは親切なトラベルアシスタントです。\n"
    if memory_context:
        system += "\n【ユーザーに関する記憶】\n"
        for i, m in enumerate(memory_context, 1): system += f"{i}. {m}\n"
        system += "\n上記の記憶を踏まえて自然に回答してください。"

    print(f"  📋 注入したメモリ ({len(memory_context)} 件):")
    for m in memory_context: print(f"    - {m}")

    resp = requests.post(f"http://127.0.0.1:{LLM_PORT}/v1/chat/completions", json={
        "model": LLM_MODEL_ID,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user_question}
        ],
        "max_tokens": 250,
        "temperature": 0.7
    }, timeout=INFER_TIMEOUT).json()
    return resp["choices"][0]["message"]["content"]

for q in [
    "ニューヨーク旅行の計画を手伝ってください。",
    "私の食の好みに合ったレストランを提案してください。",
]:
    print("\n" + "="*60)
    print(f"👤 {q}")
    print(f"\n🤖 {textwrap.fill(chat_with_memory(q), 56)}")

## 👥 Step 13: 複数ユーザーのメモリ分離デモ

In [ ]:
memory_bob = project.memory(
    group_id="default", agent_id="travel_agent",
    user_id="bob", session_id="session_001"
)
for text, meta in [
    ("私はボブです。大阪在住です。",           {"category": "profile"}),
    ("飛行機はビジネスクラスを好みます。",     {"category": "travel_pref"}),
    ("ベジタリアンなので肉料理は食べません。", {"category": "food"}),
]:
    memory_bob.add(text, metadata=meta)
    print(f"✅ Bob: {text}")

print("\n" + "="*60)
q = "旅行時のフライトの好みは？"
print(f"クエリ: '{q}'")
for name, mem in [("Alice", memory_alice), ("Bob", memory_bob)]:
    print(f"\n  [{name}]")
    for m in get_memories(mem, q)[:2]:
        print(f"    📌 {m}")

## 🧹 Step 14: クリーンアップ（任意）

In [ ]:
# サービスを停止したい場合はコメントを外して実行
# proc_llm.terminate(); print("LLM サーバー 停止")
# proc_emb.terminate(); print("Embedding サーバー 停止")
# proc_mm.terminate();  print("MemMachine サーバー 停止")
# subprocess.run("neo4j stop", shell=True)

mode = "CPU" if not HAS_GPU else f"GPU ({CUDA_VER})"
print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ デモ完了（{mode} モード）

構成:
  ユーザー（ノートブック）
      ↓  memmachine-client (timeout={600 if not HAS_GPU else 300}s)
  MemMachine サーバー  (127.0.0.1:{MM_PORT})
      ├─ PostgreSQL + pgvector → セマンティックメモリ
      ├─ Neo4j                 → エピソードメモリ
      ├─ llama-cpp-python ({LLM_PORT}) → Llama-3.2-1B 推論
      └─ llama-cpp-python ({EMB_PORT}) → nomic-embed-text 埋め込み

ポート割り当て:
  {MM_PORT}  → MemMachine (環境変数 PORT で明示指定)
  {LLM_PORT}  → llama.cpp LLM
  {EMB_PORT}  → llama.cpp Embedding
  8080 → 使用しない（システム予約の可能性があるため）
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")